In [1]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

In [2]:
# display = print
# HTML = lambda x: x

In [3]:
original_distribution = pd.read_csv(
    "~/Box/dsi-core/11th-hour/good-food-purchasing/CONFIDENTIAL_GFPP Product Attribute List_8.26.25.csv",
    dtype=str,
)[
    [
        "Level of Processing",
    ]
].dropna(
    subset="Level of Processing"
)
original_distribution["Level of Processing"] = original_distribution["Level of Processing"].map(
    {
        "Whole/Minimally Processed": 1,
        "Culinary Ingredient": 2,
        "Moderately Processed": 3,
        "Ultra-Processed": 4,
    }
)
original_distribution = original_distribution[original_distribution["Level of Processing"].notna()]
original_distribution["Level of Processing"] = original_distribution["Level of Processing"].astype(int)
original_distribution = original_distribution["Level of Processing"].value_counts()
original_distribution

Level of Processing
4    44109
1    30744
3     3653
2     3532
Name: count, dtype: int64

In [4]:
df = pd.read_csv("~/Box/dsi-core/11th-hour/good-food-purchasing/cgfp-training-with-category-try1/cgfp-test-results.csv")

In [5]:
wrong_distribution = df["Level of Processing"].value_counts()
wrong_distribution

Level of Processing
4    40670
1    27306
3      221
2      100
Name: count, dtype: int64

In [6]:
weights = (original_distribution / original_distribution.loc[2]) / (wrong_distribution / wrong_distribution.loc[2])
weights

Level of Processing
4    0.030707
1    0.031877
3    0.467990
2    1.000000
Name: count, dtype: float64

In [7]:
dict(weights)

{4: np.float64(0.030706643339020446),
 1: np.float64(0.031877304479022574),
 3: np.float64(0.46799014056358673),
 2: np.float64(1.0)}

In [8]:
df["weight"] = df["Level of Processing"].map(dict(weights))

In [9]:
df

,index,Food Product Category,Primary Food Product Category,Level of Processing,message,prob1,prob2,prob3,prob4,weight
0,71223,Pork,Pork,4,SEABOARD FOODS\n\nBACON CHIP CUBED REGULAR .37...,6.933905e-06,2.251109e-06,9.959206e-01,4.070099e-03,0.030707
1,75247,Vegetables,Vegetables,1,V-PACKER\n\nMIXED VEGETABLES\n(Vegetables),9.464380e-01,6.251838e-05,4.712037e-02,6.377049e-03,0.031877
2,11104,Condiments & Snacks,Condiments & Snacks,4,AMERICANA\n\nMAPLE SYRUP PACKETS\n(Condiments ...,1.621223e-08,1.000000e+00,7.194133e-09,1.955568e-08,0.030707
3,78829,Roots & Tubers,Roots & Tubers,1,GRIMMWAY FARMS\nGRIMMWAY FARMS\nCARROTS SHREDD...,1.000000e+00,1.262610e-08,5.964146e-09,1.621223e-08,0.031877
4,49291,Vegetables,Vegetables,1,"IOTT RANCH & ORCHARD\n\nCABBAGE, SHRED, RED, 5...",1.000000e+00,0.000000e+00,0.000000e+00,2.215949e-08,0.031877
...,...,...,...,...,...,...,...,...,...,...
68292,60380,Condiments & Snacks,Condiments & Snacks,4,The Posh Bakery\n\nPKG CROISSANT STANDARD CHOC...,3.581748e-10,6.691586e-10,4.691164e-08,9.999998e-01,0.030707
68293,35433,Chicken,Chicken,1,GEORGE'S INC\n\nCHICKEN CVP BRST B/S 5 OZ FZ\n...,3.733808e-01,2.466422e-05,8.331251e-02,5.432658e-01,0.031877
68294,56737,Meals,Cheese,4,"ELLIO'S\n\nPIZZA, CHS 2.03 Z PRSNL FZN\n(Meals)",2.594609e-11,6.162738e-12,4.691164e-08,1.000000e+00,0.030707
68295,22850,Fruit,Fruit,1,LA Foods\n\nPEACHES DICED LS10-PEAC-DIC-15372\...,2.737634e-01,1.336226e-04,5.795572e-01,1.465350e-01,0.031877


In [10]:
len(df)

68297

In [11]:
df["total_prob"] = df["prob1"] + df["prob2"] + df["prob3"] + df["prob4"]
df[df["total_prob"] < 0.99].sort_values("total_prob", ascending=False)

,index,Food Product Category,Primary Food Product Category,Level of Processing,message,prob1,prob2,prob3,prob4,weight,total_prob
6934,56713,Grain Products,Grain Products,4,"PIANTEDOSI\n\nROLL, PTATO 2.5 SQ SLCD BKD\n(G...",0.000002,0.000001,0.000122,0.98889,0.030707,0.989014


In [12]:
def confusion_count(df, normalize=True):
    matrix = np.zeros((4, 4), dtype=float)
    for i in range(4):
        selected = df[df["Level of Processing"] == i + 1]
        denominator = np.sum(selected["weight"])
        bests = np.argmax(selected[["prob1", "prob2", "prob3", "prob4"]].to_numpy(), axis=1)
        for j in range(4):
            if normalize:
                if denominator != 0:
                    matrix[i, j] = np.sum(selected[bests == j]["weight"]) / denominator
            else:
                matrix[i, j] = np.sum(selected[bests == j]["weight"])
    return matrix

def precision(matrix):
    return np.diagonal(matrix) / matrix.sum(axis=0)

In [13]:
def confusion_HTML(matrix, decimal_places=3, colorize=True):
    display(HTML(f"""

<table>
  <tr><td style="text-align: right;"><b>model predicts</b></td>{
      ''.join(f'<td style="text-align: right;"><b>{i}</b></td>' for i in range(1, 5))
  }
  {''.join(
      f'<tr><td style="text-align: right;"><b>{"truth is " if j == 1 else ""}{j}</b></td>'
      + ''.join(f'<td style="text-align: right; background-color: #{
          int(round(256 * (1 - matrix[j - 1, i - 1]))) if colorize else 255:02x
      }ffff">{
          round(matrix[j - 1, i - 1], decimal_places)
      }</td>' for i in range(1, 5))
      + '</tr>' for j in range(1, 5))}
</table>

"""))

In [14]:
display(HTML("Count in each confusion matrix entry, weighted to reproduce the original distribution of NOVA categories:"))
confusion_HTML(confusion_count(df, normalize=False), colorize=False)
display(HTML("<br>"))

display(HTML("Same, normalized by truth category (for computing the accuracy):"))
confusion_HTML(confusion_count(df))
display(HTML("<br>"))

display(HTML(f"Precision when model predicts {
    ' '.join(f'<span style="margin-left: 10px;"><b>{i + 1}</b>: {x * 100:.0f}%</span>'
            for i, x in enumerate(precision(confusion_count(df, normalize=False))))
}"))

model predicts,1,2,3,4
truth is 1,794.829,15.11,31.973,28.53
2,0.0,93.0,2.0,5.0
3,7.488,0.936,75.346,19.656
4,55.088,41.822,115.027,1036.902


model predicts,1,2,3,4
truth is 1,0.913,0.017,0.037,0.033
2,0.0,0.93,0.02,0.05
3,0.072,0.009,0.729,0.19
4,0.044,0.033,0.092,0.83


In [15]:
df["Food Product Category"].value_counts()

Food Product Category
Condiments & Snacks      16159
Vegetables                8621
Meals                     7224
Fruit                     6444
Grain Products            5655
Beverages                 5354
Roots & Tubers            3226
Chicken                   2484
Beef                      2449
Pork                      1756
Cheese                    1633
Milk & Dairy              1106
Turkey, Other Poultry     1101
Milk                       911
Yogurt                     852
Seafood                    750
Rice                       446
Eggs                       436
Tree Nuts & Seeds          430
Meat                       427
Legumes                    424
Fish (Wild)                235
Fish (Farm-Raised)         105
Produce                     43
Butter                      12
Fish (Farm-raised)           4
Fish (Farmed-Raised)         2
Fish (Wild)                  2
Chicken                      1
fruit                        1
pork                         1
Meals            

In [16]:
consolidation = {
    "Roots & Tubers": "Roots, Tubers, Legumes & Rice",
    "Milk & Dairy": "Milk, Dairy & Eggs",
    "Cheese": "Milk, Dairy & Eggs",
    "Milk": "Milk, Dairy & Eggs",
    "Yogurt": "Milk, Dairy & Eggs",
    "Legumes": "Roots, Tubers, Legumes & Rice",
    "Eggs": "Milk, Dairy & Eggs",
    "Fish (Wild)": "Seafood",
    "Fish (Farm-Raised)": "Seafood",
    "Produce": "Vegetables",
    "Butter": "Milk, Dairy & Eggs",
    "Fish (Farm-raised)": "Seafood",
    "Fish (Farmed-Raised)": "Seafood",
    "Fish (Wild) ": "Seafood",
    "meals": "Meals",
    "Meals ": "Meals",
    "Fish (Wild-Caught)": "Seafood",
    "Milk & Dairy ": "Milk, Dairy & Eggs",
    "Chicken ": "Chicken",
    "Rice": "Roots, Tubers, Legumes & Rice",
    "Tree Nuts & Seeds": "Roots, Tubers, Legumes & Rice",
    "fruit": "Fruit",
    "Chicken": "Meat",
    "Beef": "Meat",
    "Pork": "Meat",
    "Chicken ": "Meat",
    "Turkey, Other Poultry": "Meat",
    "Pork": "Meat",
    "pork": "Meat",
}

df["category"] = df["Food Product Category"].map(lambda x: consolidation.get(x, x))
categories = list(df["category"].value_counts().index)
df["category"].value_counts()

category
Condiments & Snacks              16159
Vegetables                        8664
Meat                              8219
Meals                             7225
Fruit                             6445
Grain Products                    5655
Beverages                         5354
Milk, Dairy & Eggs                4951
Roots, Tubers, Legumes & Rice     4526
Seafood                           1099
Name: count, dtype: int64

In [17]:
for category in categories:
    selected = df.query(f"category == {category!r}")
    counts = ", ".join(f"{np.count_nonzero(selected["Level of Processing"] == i)}" for i in range(1, 5))

    display(HTML(f"<b>Category:</b> {category} has {len(selected)} instances ({counts} in each row)"))
    confusion_HTML(confusion_count(selected))
    display(HTML(f"Precision when model predicts {
        ' '.join(f'<span style="margin-left: 10px;"><b>{i + 1}</b>: {x * 100:.0f}%</span>'
        for i, x in enumerate(precision(confusion_count(selected, normalize=False))))
    }"))

    display(HTML("<br>"))

model predicts,1,2,3,4
truth is 1,0.698,0.25,0.04,0.013
2,0.0,0.922,0.022,0.056
3,0.015,0.029,0.529,0.426
4,0.003,0.085,0.075,0.836


model predicts,1,2,3,4
truth is 1,0.965,0.0,0.034,0.001
2,0.0,0.0,0.0,0.0
3,0.091,0.0,0.909,0.0
4,0.345,0.0,0.603,0.052


/tmp/ipykernel_61702/2211600216.py:16: RuntimeWarning: invalid value encountered in divide
  return np.diagonal(matrix) / matrix.sum(axis=0)


model predicts,1,2,3,4
truth is 1,0.869,0.0,0.034,0.097
2,0.0,0.0,0.0,0.0
3,0.0,0.0,0.25,0.75
4,0.098,0.0,0.19,0.712


model predicts,1,2,3,4
truth is 1,0.148,0.0,0.2,0.652
2,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0
4,0.004,0.0,0.014,0.982


...

model predicts,1,2,3,4
truth is 1,0.853,0.007,0.004,0.137
2,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,1.0
4,0.128,0.017,0.01,0.845


model predicts,1,2,3,4
truth is 1,0.963,0.007,0.009,0.022
2,0.0,1.0,0.0,0.0
3,0.096,0.0,0.822,0.082
4,0.05,0.012,0.234,0.705


model predicts,1,2,3,4
truth is 1,0.958,0.001,0.038,0.003
2,0.0,0.0,0.0,0.0
3,0.062,0.0,0.937,0.0
4,0.173,0.046,0.323,0.458


model predicts,1,2,3,4
truth is 1,0.867,0.0,0.125,0.009
2,0.0,0.0,0.0,0.0
3,0.5,0.0,0.5,0.0
4,0.182,0.0,0.21,0.608


/tmp/ipykernel_61702/2211600216.py:16: RuntimeWarning: invalid value encountered in divide
  return np.diagonal(matrix) / matrix.sum(axis=0)
